# Lesson 3: Understand causal self-attention

Build queries, keys, values, and a causal mask, then verify that future tokens cannot affect earlier outputs.

**How to run:** Select a Python kernel with PyTorch installed, then run each code cell from top to bottom with **Shift+Enter**. This notebook is self-contained; no other notebook needs to run first. Restart the kernel and run from the top to reset the experiment.

**Source:** This lesson was developed from the [reference conversation's roadmap](https://chatgpt.com/share/6aa56bca-4a1c-83e9-9153-1edcc7ff7e40). The reference supplies Lesson 1 and a topic outline; Lessons 2–12 are newly written implementations of those topics. Small examples demonstrate the mechanics; they are not trained assistants.


In [ ]:
import math
from pathlib import Path
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)
# Small tensors can be slower with many CPU threads.
torch.set_num_threads(1)
device = torch.device('cpu')
print('PyTorch:', torch.__version__, '| device:', device)


## Begin with token vectors

For this experiment, random vectors stand in for embeddings. A query describes what a position seeks; a key describes what a position offers; a value carries the information to mix. Learned projections create these vectors.


In [ ]:
B, T, C = 1, 4, 8
x = torch.randn(B, T, C)
head_size = 4
query = nn.Linear(C, head_size, bias=False)
key = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
q, k, v = query(x), key(x), value(x)
print('Q, K, V:', q.shape, k.shape, v.shape)


## Compare queries with keys

The matrix entry at row `i`, column `j` measures how much position `i` attends to position `j`. Division by the square root of head size helps control score magnitudes. Softmax acts across the key positions.


In [ ]:
scores = q @ k.transpose(-2, -1) / math.sqrt(head_size)
mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
masked_scores = scores.masked_fill(~mask, float('-inf'))
weights = masked_scores.softmax(dim=-1)
out = weights @ v
print('Attention weights:', weights[0].detach().round(decimals=3))
print('Row sums:', weights.sum(dim=-1))
print('Output shape:', out.shape)
assert torch.allclose(weights.sum(dim=-1), torch.ones(B, T))
assert torch.count_nonzero(weights[0].triu(diagonal=1)) == 0


## Generalize to multiple heads

Heads learn separate comparisons in parallel. Concatenate their results, then mix them with a learned projection.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, width, heads, context_length, dropout=0.0):
        super().__init__()
        assert width % heads == 0
        self.heads = heads
        self.head_size = width // heads
        self.qkv = nn.Linear(width, 3 * width, bias=False)
        self.projection = nn.Linear(width, width)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('causal_mask', torch.tril(torch.ones(context_length, context_length, dtype=torch.bool)))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # Give each head its own vector slice: [B, heads, T, head_size].
        q = q.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        k = k.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        v = v.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_size)
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float('-inf'))
        weights = self.dropout(F.softmax(scores, dim=-1))
        out = (weights @ v).transpose(1, 2).contiguous().reshape(B, T, C)
        return self.projection(out)


## Check causality by changing the future

Change only the final token. Earlier outputs must stay equal; the final output may change.


In [ ]:
attention = CausalSelfAttention(C, heads=2, context_length=T)
attention.eval()
changed = x.clone()
changed[:, -1] += torch.arange(C, dtype=x.dtype) * 10
with torch.no_grad():
    original_out = attention(x)
    changed_out = attention(changed)
assert torch.allclose(original_out[:, :-1], changed_out[:, :-1], atol=1e-6)
print('Earlier positions unchanged:', True)
print('Final position difference:', (original_out[:, -1] - changed_out[:, -1]).abs().max().item())


## Try it yourself

Inspect the first row of the attention matrix. Why must its first entry be 1? Remove the mask in an experimental copy and repeat the future-token test. Try 1, 2, and 4 heads while keeping width 8.
